# Digit Recognition: CNN vs Vision Transformer

This notebook tackles the [Digit Recognizer](https://www.kaggle.com/competitions/digit-recognizer) competition on Kaggle.
We compare two approaches:

1. **CNN Baseline** -- A compact convolutional neural network trained from scratch.
2. **Vision Transformer (ViT)** -- A ViT-Tiny model from the `timm` library, adapted for grayscale 28x28 input.

Both models are trained for 5 epochs and evaluated on a hold-out validation split.
The best model is used to generate `submission.csv`.

In [ ]:
# ============================================================
# Cell 1 -- Imports
# ============================================================
import os
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

import torchvision.transforms as T

import timm
from sklearn.model_selection import train_test_split

# Reproducibility
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

In [ ]:
# ============================================================
# Cell 2 -- Load Data
# ============================================================
DATA_DIR = "/kaggle/input/digit-recognizer"

train_df = pd.read_csv(os.path.join(DATA_DIR, "train.csv"))
test_df  = pd.read_csv(os.path.join(DATA_DIR, "test.csv"))

# Separate labels and pixel values
labels = train_df["label"].values
pixels = train_df.drop(columns=["label"]).values.astype(np.float32)
test_pixels = test_df.values.astype(np.float32)

# Reshape to 28x28 images
images = pixels.reshape(-1, 28, 28)
test_images = test_pixels.reshape(-1, 28, 28)

print(f"Training samples : {images.shape[0]}")
print(f"Test samples     : {test_images.shape[0]}")
print(f"Label distribution: {np.bincount(labels)}")

In [ ]:
# ============================================================
# Cell 3 -- Train / Validation Split
# ============================================================
VAL_SIZE = 0.1

X_train, X_val, y_train, y_val = train_test_split(
    images, labels, test_size=VAL_SIZE, random_state=SEED, stratify=labels
)

print(f"Train: {X_train.shape[0]}  |  Val: {X_val.shape[0]}")

In [ ]:
# ============================================================
# Cell 4 -- Custom Dataset
# ============================================================
class MNISTDataset(Dataset):
    """PyTorch Dataset for MNIST images loaded from CSV."""

    def __init__(self, images: np.ndarray, labels: np.ndarray = None, transform=None):
        """
        Args:
            images:    numpy array of shape (N, 28, 28), pixel values in [0, 255].
            labels:    numpy array of shape (N,) or None for test data.
            transform: torchvision transforms to apply to each image.
        """
        self.images = images
        self.labels = labels
        self.transform = transform

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        # Convert to (1, 28, 28) float tensor in [0, 1]
        img = self.images[idx] / 255.0
        img = torch.tensor(img, dtype=torch.float32).unsqueeze(0)  # (1, H, W)

        if self.transform:
            img = self.transform(img)

        if self.labels is not None:
            label = torch.tensor(self.labels[idx], dtype=torch.long)
            return img, label
        return img


# Transforms -------------------------------------------------------
# CNN: keep 28x28
cnn_transform = None  # images are already normalised in __getitem__

# ViT: resize to 224x224 and repeat channels (timm ViT expects 3-ch)
vit_transform = T.Compose([
    T.Resize((224, 224), antialias=True),
    T.Lambda(lambda x: x.repeat(3, 1, 1)),      # 1-ch -> 3-ch
    T.Normalize(mean=[0.485, 0.456, 0.406],
                std=[0.229, 0.224, 0.225]),
])

BATCH_SIZE = 128


def get_loaders(transform):
    """Return train, val, and test DataLoaders for the given transform."""
    train_ds = MNISTDataset(X_train, y_train, transform=transform)
    val_ds   = MNISTDataset(X_val,   y_val,   transform=transform)
    test_ds  = MNISTDataset(test_images, labels=None, transform=transform)

    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                              num_workers=2, pin_memory=True)
    val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False,
                              num_workers=2, pin_memory=True)
    test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False,
                              num_workers=2, pin_memory=True)
    return train_loader, val_loader, test_loader

print("Dataset and DataLoader helpers ready.")

In [ ]:
# ============================================================
# Cell 5 -- Visualize Sample Digits
# ============================================================
fig, axes = plt.subplots(2, 8, figsize=(14, 4))
for i, ax in enumerate(axes.flat):
    ax.imshow(X_train[i], cmap="gray")
    ax.set_title(f"Label: {y_train[i]}", fontsize=9)
    ax.axis("off")
plt.suptitle("Sample Training Digits", fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# Cell 6 -- Training & Evaluation Utilities
# ============================================================
def train_one_epoch(model, loader, criterion, optimizer):
    """Train for one epoch; return average loss."""
    model.train()
    running_loss = 0.0
    for imgs, labels in loader:
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        logits = model(imgs)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * imgs.size(0)
    return running_loss / len(loader.dataset)


@torch.no_grad()
def evaluate(model, loader):
    """Evaluate model; return (loss, accuracy)."""
    model.eval()
    criterion = nn.CrossEntropyLoss()
    total_loss, correct, total = 0.0, 0, 0
    for imgs, labels in loader:
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        logits = model(imgs)
        total_loss += criterion(logits, labels).item() * imgs.size(0)
        preds = logits.argmax(dim=1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)
    return total_loss / total, correct / total


def train_model(model, train_loader, val_loader, epochs=5, lr=1e-3):
    """Full training loop. Returns history dict."""
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=lr)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)

    history = {"train_loss": [], "val_loss": [], "val_acc": []}

    for epoch in range(1, epochs + 1):
        t0 = time.time()
        train_loss = train_one_epoch(model, train_loader, criterion, optimizer)
        val_loss, val_acc = evaluate(model, val_loader)
        scheduler.step()
        elapsed = time.time() - t0

        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        history["val_acc"].append(val_acc)

        print(f"Epoch {epoch}/{epochs}  "
              f"train_loss={train_loss:.4f}  "
              f"val_loss={val_loss:.4f}  "
              f"val_acc={val_acc:.4f}  "
              f"({elapsed:.1f}s)")

    return history

print("Training utilities ready.")

In [ ]:
# ============================================================
# Cell 7 -- CNN Baseline
# ============================================================
class SimpleCNN(nn.Module):
    """Two conv-layer CNN for 28x28 grayscale digit classification."""

    def __init__(self, num_classes=10):
        super().__init__()
        self.features = nn.Sequential(
            # Block 1
            nn.Conv2d(1, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),                          # -> 32 x 14 x 14
            # Block 2
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),                          # -> 64 x 7 x 7
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64 * 7 * 7, 128),
            nn.ReLU(inplace=True),
            nn.Dropout(0.3),
            nn.Linear(128, num_classes),
        )

    def forward(self, x):
        return self.classifier(self.features(x))


# Build loaders (no extra transform for CNN)
cnn_train_loader, cnn_val_loader, cnn_test_loader = get_loaders(transform=None)

# Train
cnn_model = SimpleCNN().to(DEVICE)
print(f"CNN parameters: {sum(p.numel() for p in cnn_model.parameters()):,}")
cnn_history = train_model(cnn_model, cnn_train_loader, cnn_val_loader, epochs=5, lr=1e-3)

In [ ]:
# ============================================================
# Cell 8 -- Vision Transformer (ViT-Tiny via timm)
# ============================================================
# Load a pre-trained ViT-Tiny and adapt the classifier head.
vit_model = timm.create_model(
    "vit_tiny_patch16_224",
    pretrained=True,
    num_classes=10,
)
vit_model = vit_model.to(DEVICE)

print(f"ViT parameters: {sum(p.numel() for p in vit_model.parameters()):,}")

# Build loaders with ViT transform (resize + 3-ch + normalise)
vit_train_loader, vit_val_loader, vit_test_loader = get_loaders(transform=vit_transform)

# Fine-tune with a smaller learning rate since we start from pretrained weights
vit_history = train_model(vit_model, vit_train_loader, vit_val_loader, epochs=5, lr=3e-4)

In [ ]:
# ============================================================
# Cell 9 -- Compare CNN vs ViT
# ============================================================
epochs_range = range(1, 6)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# -- Loss --
ax = axes[0]
ax.plot(epochs_range, cnn_history["train_loss"], "o-", label="CNN train")
ax.plot(epochs_range, cnn_history["val_loss"],   "s--", label="CNN val")
ax.plot(epochs_range, vit_history["train_loss"], "o-", label="ViT train")
ax.plot(epochs_range, vit_history["val_loss"],   "s--", label="ViT val")
ax.set_xlabel("Epoch")
ax.set_ylabel("Loss")
ax.set_title("Training & Validation Loss")
ax.legend()
ax.grid(True, alpha=0.3)

# -- Accuracy --
ax = axes[1]
ax.plot(epochs_range, cnn_history["val_acc"], "o-", label="CNN val acc")
ax.plot(epochs_range, vit_history["val_acc"], "s-", label="ViT val acc")
ax.set_xlabel("Epoch")
ax.set_ylabel("Accuracy")
ax.set_title("Validation Accuracy")
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Summary table
print("\n" + "=" * 50)
print(f"{'Model':<10} {'Best Val Acc':>12}  {'Final Train Loss':>16}")
print("-" * 50)
print(f"{'CNN':<10} {max(cnn_history['val_acc']):>12.4f}  {cnn_history['train_loss'][-1]:>16.4f}")
print(f"{'ViT':<10} {max(vit_history['val_acc']):>12.4f}  {vit_history['train_loss'][-1]:>16.4f}")
print("=" * 50)

In [ ]:
# ============================================================
# Cell 10 -- Generate Submission
# ============================================================
# Pick the model with the higher validation accuracy
best_cnn_acc = max(cnn_history["val_acc"])
best_vit_acc = max(vit_history["val_acc"])

if best_vit_acc >= best_cnn_acc:
    print("Using ViT for submission.")
    best_model = vit_model
    best_test_loader = vit_test_loader
else:
    print("Using CNN for submission.")
    best_model = cnn_model
    best_test_loader = cnn_test_loader


@torch.no_grad()
def predict(model, loader):
    """Generate predictions for unlabelled test data."""
    model.eval()
    all_preds = []
    for batch in loader:
        # Test dataset returns images only (no labels)
        imgs = batch.to(DEVICE) if isinstance(batch, torch.Tensor) else batch[0].to(DEVICE)
        logits = model(imgs)
        all_preds.append(logits.argmax(dim=1).cpu())
    return torch.cat(all_preds).numpy()


predictions = predict(best_model, best_test_loader)

submission = pd.DataFrame({
    "ImageId": np.arange(1, len(predictions) + 1),
    "Label": predictions,
})
submission.to_csv("submission.csv", index=False)

print(f"Submission saved -- {len(submission)} rows")
print(submission.head(10))

# Insights: CNN vs Vision Transformer

## When to use a CNN

- **Small datasets.** CNNs have strong inductive biases (locality, translation equivariance) that let them learn effectively from fewer samples.
- **Limited compute.** The CNN baseline above has roughly 400K parameters and trains in seconds per epoch on a GPU. Ideal for quick iteration.
- **Fixed, small input sizes.** For 28x28 images the receptive field of a small CNN already covers the entire image.

## When to use a Vision Transformer

- **Larger datasets or pre-training available.** ViTs shine when pre-trained on large corpora (ImageNet-21K, etc.) and fine-tuned on the target task.
- **Higher-resolution or complex images.** Self-attention captures long-range dependencies that CNNs need deep stacks to achieve.
- **State-of-the-art accuracy matters more than latency.** ViTs typically achieve higher accuracy at the cost of more FLOPs.

## Trade-offs observed in this notebook

| Aspect | CNN | ViT-Tiny |
|--------|-----|----------|
| Parameters | ~400K | ~5.7M |
| Training speed | Fast (native 28x28) | Slower (resize to 224x224) |
| Accuracy (5 epochs) | High | Comparable or higher (benefits from pre-training) |
| Data efficiency | Excellent on small data | Needs pre-training to match |

**Key takeaway:** For simple tasks like MNIST, a well-tuned CNN is hard to beat on efficiency. ViTs become increasingly attractive as image complexity and dataset size grow, especially when leveraging pre-trained weights.